# Bayesian Inference: Theoretical Foundations and Computational Methods

## 1. Introduction and Overview

Bayesian Inference represents a profound philosophical and mathematical shift from traditional frequentist statistics. Instead of treating model parameters as fixed, unknown constants, Bayesian inference treats them as random variables governed by probability distributions. 

This paradigm shift allows for the explicit quantification of uncertainty, the incorporation of prior domain knowledge, and the computation of direct probabilistic statements about parameters (e.g., "There is a 95% probability that the conversion rate lies between 4% and 6%"), which is strictly prohibited in the frequentist framework.

This notebook provides a rigorous, hands-on guide to Bayesian Inference. We will explore belief updating, Bayes Theorem, Grid Approximation, Conjugate Priors, and practical applications like Bayesian A/B testing.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Configure pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Configure visualization styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("deep")
warnings.filterwarnings('ignore')

# Set global random seed for complete reproducibility
np.random.seed(42)

print("Environment successfully initialized. Random seed set to 42.")

## 3. The Philosophy: Belief vs. Frequency

The Frequentist View: "The drug's effectiveness is a fixed, unknown value. If I repeat this clinical trial 1,000 times, my confidence interval will capture the true value 95% of the time."

The Bayesian View: "I have an initial belief about the drug's effectiveness based on past trials (the Prior). As I see new data from this trial (the Likelihood), I update my belief (the Posterior). My result is a probability distribution of the drug's effectiveness."

Bayes Theorem is the mathematical formalization of this learning process:

Posterior = (Likelihood * Prior) / Marginal Evidence
P(theta | X) = P(X | theta) * P(theta) / P(X)

Because the marginal evidence P(X) is just a normalizing constant, we often focus on the proportional relationship:
Posterior is proportional to Likelihood * Prior

## 4. Data Creation: Pharmaceutical Clinical Trial

Let us simulate a scenario where we are testing the efficacy of a new drug. The outcome is binary: a patient either recovers (1) or does not recover (0). We will conduct a small clinical trial.

In [ ]:
# Generate synthetic clinical trial data
n_patients = 20
true_efficacy = 0.70  # The hidden truth we are trying to estimate

# Simulate patient outcomes using a Binomial distribution
# 1 = recovered, 0 = not recovered
trial_data = np.random.binomial(n=1, p=true_efficacy, size=n_patients)

recoveries = np.sum(trial_data)
failures = n_patients - recoveries
raw_success_rate = recoveries / n_patients

print("--- Clinical Trial Results ---")
print(f"Total Patients Tested: {n_patients}")
print(f"Recoveries (Successes): {recoveries}")
print(f"Failures: {failures}")
print(f"Raw Empirical Success Rate: {raw_success_rate:.2f}")

## 5. Core Concept 1: Grid Approximation

Before diving into complex analytical formulas, we can compute the Posterior distribution using a discrete grid. We evaluate our prior belief and the likelihood of the data at a finite number of points between 0 and 1.

1. Define a grid of possible parameter values (e.g., 0.00, 0.01, ..., 1.00).
2. Define the Prior probability for each grid value.
3. Compute the Likelihood of our observed data at each grid value.
4. Multiply Prior by Likelihood, then normalize to sum to 1 to obtain the Posterior.

In [ ]:
# 1. Define the grid (100 points between 0 and 1)
p_grid = np.linspace(0, 1, 100)

# 2. Define a Uniform Prior (we believe all efficacies are equally likely initially)
prior = np.ones(100) / 100

# 3. Compute Likelihood using the Binomial PMF
# Probability of getting exactly 'recoveries' out of 'n_patients' given parameter 'p'
likelihood = stats.binom.pmf(k=recoveries, n=n_patients, p=p_grid)

# 4. Compute unnormalized posterior, then normalize
unnormalized_posterior = likelihood * prior
posterior = unnormalized_posterior / np.sum(unnormalized_posterior)

print("Grid approximation computation complete.")
max_posterior_idx = np.argmax(posterior)
print(f"Maximum Posterior Probability (MAP) occurs at p = {p_grid[max_posterior_idx]:.2f}")

## 6. Visualizing the Grid Approximation

Let us plot the Prior, Likelihood, and Posterior side-by-side to visually grasp how Bayesian updating incorporates new evidence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot Prior
axes[0].plot(p_grid, prior, color='blue', linewidth=2)
axes[0].set_title('Prior Distribution (Our initial belief)')
axes[0].set_xlabel('Probability of Efficacy (p)')
axes[0].set_ylabel('Belief')
axes[0].set_ylim(0, np.max(posterior) * 1.1)

# Plot Likelihood
axes[1].plot(p_grid, likelihood, color='green', linewidth=2)
axes[1].set_title('Likelihood (The Evidence)')
axes[1].set_xlabel('Probability of Efficacy (p)')

# Plot Posterior
axes[2].plot(p_grid, posterior, color='purple', linewidth=2)
axes[2].set_title('Posterior Distribution (Updated belief)')
axes[2].set_xlabel('Probability of Efficacy (p)')

plt.tight_layout()
plt.show()

print("Notice how the perfectly flat Prior is molded by the bell-shaped Likelihood into the Posterior.")

## 7. Core Concept 2: Conjugate Priors (Analytical Solutions)

Grid approximation is intuitive but computationally expensive for complex models. When our Likelihood function is Binomial, we can use a Beta distribution as our Prior. 

The Beta distribution is a 'Conjugate Prior' for the Binomial likelihood. This means the Posterior will also be a Beta distribution! The mathematics simplify into basic addition of the parameters:

Prior: Beta(alpha_prior, beta_prior)
Data: successes, failures
Posterior: Beta(alpha_prior + successes, beta_prior + failures)

In [ ]:
# A Uniform prior is mathematically identical to a Beta(1, 1) distribution
alpha_prior = 1
beta_prior = 1

# Analytical Bayesian Update via Conjugate Priors
alpha_posterior = alpha_prior + recoveries
beta_posterior = beta_prior + failures

print("--- Analytical Conjugate Update Results ---")
print(f"Prior Parameters:     Beta(alpha={alpha_prior}, beta={beta_prior})")
print(f"Observed Data:        Successes={recoveries}, Failures={failures}")
print(f"Posterior Parameters: Beta(alpha={alpha_posterior}, beta={beta_posterior})")

# The expected value (mean) of a Beta(alpha, beta) distribution is alpha / (alpha + beta)
posterior_mean = alpha_posterior / (alpha_posterior + beta_posterior)
print(f"\nExpected Efficacy (Posterior Mean): {posterior_mean:.3f}")

## 8. Visualizing Conjugate Priors

Using scipy.stats, we can generate the continuous Beta distributions and visualize the exact analytical curves.

In [ ]:
x_vals = np.linspace(0, 1, 500)

# Generate analytical PDFs
prior_pdf = stats.beta.pdf(x_vals, alpha_prior, beta_prior)
posterior_pdf = stats.beta.pdf(x_vals, alpha_posterior, beta_posterior)

plt.figure(figsize=(10, 6))
plt.plot(x_vals, prior_pdf, 'b--', lw=2, label='Prior: Beta(1,1)')
plt.plot(x_vals, posterior_pdf, 'purple', lw=3, label=f'Posterior: Beta({alpha_posterior},{beta_posterior})')
plt.fill_between(x_vals, 0, posterior_pdf, color='purple', alpha=0.2)

# Mark the posterior mean
plt.axvline(posterior_mean, color='black', linestyle=':', lw=2, label=f'Posterior Mean: {posterior_mean:.2f}')

plt.title('Analytical Bayesian Updating with Conjugate Priors', fontsize=14)
plt.xlabel('Efficacy Probability (p)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Core Concept 3: The Power (and Danger) of Priors

What if we had previous domain knowledge? Imagine a related drug previously showed a 50% efficacy across thousands of patients, and we are quite confident our new drug will perform similarly. We can encode this as a 'Strong Prior', like Beta(20, 20).

Let us see how different priors (Weak vs. Strong) react to the exact same dataset.

In [ ]:
# Scenario A: Weak/Flat Prior (Beta 1, 1)
alpha_weak_post = 1 + recoveries
beta_weak_post = 1 + failures
mean_weak_post = alpha_weak_post / (alpha_weak_post + beta_weak_post)

# Scenario B: Strong Prior centered at 0.50 (Beta 20, 20)
alpha_strong_prior, beta_strong_prior = 20, 20
alpha_strong_post = alpha_strong_prior + recoveries
beta_strong_post = beta_strong_prior + failures
mean_strong_post = alpha_strong_post / (alpha_strong_post + beta_strong_post)

print("--- Prior Sensitivity Analysis ---")
print(f"Raw Data Success Rate: {raw_success_rate:.3f}")
print(f"Posterior Mean (Weak Prior):   {mean_weak_post:.3f}")
print(f"Posterior Mean (Strong Prior): {mean_strong_post:.3f}")
print("\nThe data pushes the weak prior heavily toward the empirical mean. However, the strong prior resists the new data, anchoring the posterior closer to 0.50.")

## 10. Visualizing Prior Sensitivity

Plotting the posteriors derived from different priors demonstrates how Bayesian inference balances our initial confidence against the weight of new evidence.

In [ ]:
pdf_weak_post = stats.beta.pdf(x_vals, alpha_weak_post, beta_weak_post)
pdf_strong_prior = stats.beta.pdf(x_vals, alpha_strong_prior, beta_strong_prior)
pdf_strong_post = stats.beta.pdf(x_vals, alpha_strong_post, beta_strong_post)

plt.figure(figsize=(10, 6))
# Plot the weak prior's posterior
plt.plot(x_vals, pdf_weak_post, 'b-', lw=3, label='Posterior from Weak Prior')

# Plot the strong prior and its resulting posterior
plt.plot(x_vals, pdf_strong_prior, 'r--', lw=2, label='Strong Prior: Beta(20,20)')
plt.plot(x_vals, pdf_strong_post, 'r-', lw=3, label='Posterior from Strong Prior')

# Mark the true parameter
plt.axvline(true_efficacy, color='green', linestyle=':', lw=2, label=f'True Hidden Efficacy ({true_efficacy})')

plt.title('How Prior Strength Influences the Posterior', fontsize=14)
plt.xlabel('Probability of Efficacy (p)')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Bayesian vs Frequentist Intervals

Frequentist Confidence Interval: "If we repeat the experiment infinite times, 95% of calculated intervals will contain the fixed true parameter."

Bayesian Credible Interval (Highest Density Interval): "Given this specific data and my prior, there is a 95% probability that the true parameter lies in this exact interval."

This is a massive epistemological difference. Let's calculate both.

In [ ]:
# 1. Frequentist 95% Confidence Interval (Normal Approximation)
z_score = 1.96
margin_of_error = z_score * np.sqrt((raw_success_rate * (1 - raw_success_rate)) / n_patients)
freq_ci_lower = raw_success_rate - margin_of_error
freq_ci_upper = raw_success_rate + margin_of_error

# 2. Bayesian 95% Credible Interval (using Percentile Point Function / Inverse CDF)
# We use the weak posterior for a fair comparison to the frequentist un-regularized approach
bayes_ci_lower = stats.beta.ppf(0.025, alpha_weak_post, beta_weak_post)
bayes_ci_upper = stats.beta.ppf(0.975, alpha_weak_post, beta_weak_post)

print("--- Uncertainty Intervals Comparison ---")
print(f"Frequentist 95% Confidence Interval: [{freq_ci_lower:.3f}, {freq_ci_upper:.3f}]")
print(f"Bayesian 95% Credible Interval:      [{bayes_ci_lower:.3f}, {bayes_ci_upper:.3f}]")
print("\nWhile numerically similar here, the Bayesian interval allows us to say: 'There is a 95% chance the parameter is between these two numbers'.")

## 12. Real-World Application: Bayesian A/B Testing

Let us apply Bayesian Inference to a classic business problem: A/B Testing. 
We are testing two website designs to see which yields a higher conversion rate. In the frequentist approach, we would compute a p-value. In the Bayesian approach, we will calculate the post-test probability that Variant B is better than Variant A.

In [ ]:
# Simulate A/B Test Data
n_visitors_A = 1500
n_visitors_B = 1500

# True hidden conversion rates
true_rate_A = 0.040  # 4.0% conversion
true_rate_B = 0.052  # 5.2% conversion

conversions_A = np.random.binomial(n_visitors_A, true_rate_A)
conversions_B = np.random.binomial(n_visitors_B, true_rate_B)

print("--- A/B Test Raw Results ---")
print(f"Variant A: {conversions_A} conversions / {n_visitors_A} visitors ({conversions_A/n_visitors_A*100:.2f}%)")
print(f"Variant B: {conversions_B} conversions / {n_visitors_B} visitors ({conversions_B/n_visitors_B*100:.2f}%)")

### Bayesian Monte Carlo Simulation for A/B Testing

Instead of relying on analytical formulas for the difference between two Beta distributions, we can use Monte Carlo simulation: drawing thousands of random samples from both Posterior distributions and directly comparing them.

In [ ]:
# Define Weak Priors: Beta(1,1)
prior_a, prior_b = 1, 1

# Calculate Posteriors
post_alpha_A = prior_a + conversions_A
post_beta_A = prior_b + (n_visitors_A - conversions_A)

post_alpha_B = prior_a + conversions_B
post_beta_B = prior_b + (n_visitors_B - conversions_B)

# Monte Carlo Simulation: Draw 100,000 samples from each posterior
n_mc_samples = 100000
samples_A = np.random.beta(post_alpha_A, post_beta_A, n_mc_samples)
samples_B = np.random.beta(post_alpha_B, post_beta_B, n_mc_samples)

# Calculate Probability that B is better than A
prob_B_better = np.mean(samples_B > samples_A)

# Calculate the Expected Relative Uplift
expected_uplift = np.mean((samples_B - samples_A) / samples_A) * 100

print("--- Bayesian A/B Test Insights ---")
print(f"Probability that Variant B is better than Variant A: {prob_B_better * 100:.2f}%")
print(f"Expected Relative Uplift: {expected_uplift:.2f}%")
print("\nThis is a highly actionable statement for a product manager, far more intuitive than a p-value!")

## 13. Visualizing A/B Test Posteriors

We plot the two Beta distributions to see the overlap. The less overlap, the more confident we are that the variants are truly different.

In [ ]:
x_ab = np.linspace(0.02, 0.08, 1000)
pdf_A = stats.beta.pdf(x_ab, post_alpha_A, post_beta_A)
pdf_B = stats.beta.pdf(x_ab, post_alpha_B, post_beta_B)

plt.figure(figsize=(10, 6))
plt.plot(x_ab, pdf_A, label='Variant A Posterior', color='red', lw=2)
plt.fill_between(x_ab, 0, pdf_A, alpha=0.2, color='red')

plt.plot(x_ab, pdf_B, label='Variant B Posterior', color='blue', lw=2)
plt.fill_between(x_ab, 0, pdf_B, alpha=0.2, color='blue')

plt.title('A/B Test Posterior Distributions', fontsize=14)
plt.xlabel('Conversion Rate')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Common Pitfalls: The Zero Prior

A massive trap in Bayesian inference is assigning a 0 probability to an event in your Prior. If your Prior is 0, Bayes Theorem multiplies the Likelihood by 0. No amount of evidence can ever change your mind. Let us demonstrate this mathematically using grid approximation.

In [ ]:
# Imagine a rigid prior that absolutely refuses to believe efficacy can be above 0.5
rigid_prior = np.where(p_grid <= 0.5, 2.0, 0.0)
rigid_prior = rigid_prior / np.sum(rigid_prior) # Normalize

# New overwhelming evidence: 100 successes out of 100 trials!
overwhelming_likelihood = stats.binom.pmf(k=100, n=100, p=p_grid)

# Update
rigid_unnorm_post = overwhelming_likelihood * rigid_prior
rigid_posterior = rigid_unnorm_post / np.sum(rigid_unnorm_post)

print("--- The Danger of Zero Priors ---")
print(f"Max Posterior Probability despite 100/100 successes: p = {p_grid[np.argmax(rigid_posterior)]:.2f}")
print("Lesson: Cromwell's Rule dictates you should never assign a probability of exactly 0 or 1 to any event, unless it is logically impossible.")

## 15. Practice Exercise: Manufacturing Defect Rate

Scenario: A manufacturing plant produces microchips. The defect rate is historically known to hover around 2%, acting as our prior. We can encode this as a Beta distribution with alpha=2, beta=98.

Today, a new batch of 500 chips is tested, and 15 are found defective.

Your Task:
1. Calculate the parameters for the new Posterior Beta distribution.
2. Determine the Expected Defect Rate (the mean of the posterior).

In [ ]:
# Exercise Data Setup
hist_alpha, hist_beta = 2, 98
chips_tested = 500
defects_found = 15

print(f"Historical Prior: Beta(alpha={hist_alpha}, beta={hist_beta})")
print(f"Test Data: {defects_found} defects out of {chips_tested} chips.")
print(f"Raw empirical defect rate today: {(defects_found / chips_tested) * 100:.2f}%")

### Exercise Solution

Using the conjugate prior rules for Beta-Binomial:
New Alpha = Prior Alpha + Successes (Defects in this context)
New Beta = Prior Beta + Failures (Non-defects)

In [ ]:
# Solution Calculation
post_alpha_ex = hist_alpha + defects_found
post_beta_ex = hist_beta + (chips_tested - defects_found)

expected_defect_rate = post_alpha_ex / (post_alpha_ex + post_beta_ex)

print("--- Exercise Solution ---")
print(f"1. Posterior Parameters: Beta(alpha={post_alpha_ex}, beta={post_beta_ex})")
print(f"2. Expected Defect Rate (Posterior Mean): {expected_defect_rate * 100:.2f}%")
print("\nNotice how the historical prior tempered the unusually high 3% defect rate seen in today's batch, pulling the estimate down to ~2.8%.")

## 16. Visualization Gallery 1: Sequential Updating

The beauty of Bayesian inference is that today's Posterior becomes tomorrow's Prior. Let us visualize data arriving sequentially, updating our belief step-by-step as new batches of data stream in.

In [ ]:
# True rate is 0.6. Data arrives in batches of 10.
batches = [ 
    {'success': 6, 'total': 10},
    {'success': 7, 'total': 10},
    {'success': 5, 'total': 10},
    {'success': 6, 'total': 10}
]

plt.figure(figsize=(12, 6))
x_seq = np.linspace(0, 1, 500)
current_alpha, current_beta = 1, 1 # Start Uniform

# Plot initial prior
plt.plot(x_seq, stats.beta.pdf(x_seq, current_alpha, current_beta), 'k--', label='Initial Prior', alpha=0.5)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, batch in enumerate(batches):
    current_alpha += batch['success']
    current_beta += (batch['total'] - batch['success'])
    
    y_seq = stats.beta.pdf(x_seq, current_alpha, current_beta)
    plt.plot(x_seq, y_seq, color=colors[i], lw=2, label=f'After Batch {i+1} (N={current_alpha+current_beta-2})')
    plt.fill_between(x_seq, 0, y_seq, color=colors[i], alpha=0.1)

plt.axvline(0.6, color='black', linestyle=':', lw=2, label='True Parameter (0.6)')
plt.title('Sequential Bayesian Updating (Streaming Data)', fontsize=14)
plt.xlabel('Parameter Value (p)')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("With every batch of data, the distribution becomes narrower (more confident) and hones in on the truth.")

## 17. Visualization Gallery 2: Asymptotic Convergence

The Bernstein-von Mises theorem states that as the sample size N approaches infinity, the posterior distribution converges to a Normal distribution centered at the true parameter, and the influence of the prior becomes entirely negligible.

In [ ]:
def plot_asymptotic_convergence():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    true_p_asymp = 0.65
    sample_sizes = [10, 50, 200, 1000]
    prior_a, prior_b = 5, 2  # Strong biased prior favoring high values
    
    x_asymp = np.linspace(0, 1, 500)
    prior_pdf = stats.beta.pdf(x_asymp, prior_a, prior_b)
    
    for i, n in enumerate(sample_sizes):
        succ = np.random.binomial(n, true_p_asymp)
        
        post_a = prior_a + succ
        post_b = prior_b + (n - succ)
        post_pdf = stats.beta.pdf(x_asymp, post_a, post_b)
        
        # Scale for visual comparison
        axes[i].plot(x_asymp, prior_pdf / np.max(prior_pdf), 'b--', alpha=0.5, label='Biased Prior')
        axes[i].plot(x_asymp, post_pdf / np.max(post_pdf), 'r-', linewidth=2, label='Posterior')
        axes[i].axvline(true_p_asymp, color='black', linestyle=':', label=f'True p={true_p_asymp}')
        
        axes[i].set_title(f'N = {n} (Successes = {succ})')
        axes[i].set_xlabel('Parameter p')
        axes[i].set_ylabel('Scaled Density')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.show()

plot_asymptotic_convergence()
print("Notice how at N=1000, the biased prior is completely overwhelmed by the Likelihood, and the Posterior is perfectly centered on the truth.")

## 18. Summary and Key Takeaways

- **Uncertainty as Belief**: Bayesian inference treats parameters as random variables with probability distributions, unlike the Frequentist view of fixed constants.
- **Bayes Theorem Framework**: Posterior is proportional to Likelihood * Prior. We use empirical evidence to mathematically update our initial assumptions.
- **Conjugate Priors**: Choosing mathematically compatible distributions (like Beta and Binomial) allows for exact, lightning-fast analytical solutions via simple addition.
- **Actionable Insights**: Instead of binary p-values, Bayesian methods allow us to ask direct business questions, like 'What is the exact probability that Variant B is better than Variant A?'
- **Asymptotic Behavior**: With enough data, the influence of the prior washes out, and the posterior converges to the truth (Bernstein-von Mises theorem).
- **Pitfalls**: Overly rigid priors (especially assigning 0 probability) can prevent the model from learning the truth, no matter how much data is provided.